# BrandSphere AI — Week 10: Integration Tests

End-to-end test of all modules in sequence.
This notebook simulates a full brand generation run.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.palette_engine    import generate_palette
from src.font_engine       import recommend_fonts
from src.logo_engine       import generate_all_logos
from src.slogan_engine     import generate_slogans, nltk_analyze
from src.aesthetics_engine import score_brand
from src.multilingual_engine import translate_slogan, validate_translations
from src.animation_engine  import create_brand_gif
from src.export_engine     import build_brand_kit_zip
print('✅ All modules imported successfully')

## Test Brand Config

In [ ]:
COMPANY     = 'NovaTech Solutions'
INDUSTRY    = 'Technology / Software'
PERSONALITY = 'Minimalist'
TONE        = 'Professional'
AUDIENCE    = 'Startup founders and tech teams aged 25-40'

print(f'Testing brand: {COMPANY}')
print(f'Industry:      {INDUSTRY}')
print(f'Personality:   {PERSONALITY}')

## Module 1: Color Palette (KMeans)

In [ ]:
palette = generate_palette(INDUSTRY, PERSONALITY)
print(f'Palette generated: {len(palette)} colors')
for name, v in palette.items():
    print(f'  {name:12s}: {v["hex"]}  — {v["psychology"]}')
assert len(palette) == 5, 'Expected 5 palette colors'
print('✅ Palette test passed')

## Module 2: Font Recommendations

In [ ]:
fonts = recommend_fonts(INDUSTRY, PERSONALITY)
print(f'Fonts recommended: {len(fonts)}')
for f in fonts:
    print(f'  {f["rank"]}: {f["heading"]} + {f["body"]}')
assert len(fonts) == 3, 'Expected 3 font recommendations'
print('✅ Font test passed')

## Module 3: Logo Generation

In [ ]:
logos = generate_all_logos(COMPANY, palette)
print(f'Logos generated: {len(logos)}')
for l in logos:
    assert l['svg'].startswith('<svg'), 'SVG should start with <svg'
    print(f'  {l["style"]}: {len(l["svg"])} chars SVG')
assert len(logos) == 5, 'Expected 5 logo concepts'
print('✅ Logo test passed')

## Module 4: Slogan Engine (NLTK + TF-IDF)

In [ ]:
slogans, retrieved = generate_slogans(COMPANY, INDUSTRY, TONE, AUDIENCE)
print(f'Slogans generated: {len(slogans)}')
for s in slogans:
    print(f'  [{s["source"]:8s}] "{s["text"]}"')
assert len(slogans) > 0, 'Expected at least 1 slogan'

# NLTK analysis
analysis = slogans[0]['analysis']
print(f'\nNLTK Analysis of first slogan:')
print(f'  Tokens: {analysis["token_count"]}')
print(f'  Clean:  {analysis["clean_tokens"]}')
print(f'  Stems:  {analysis["stems"]}')
print('✅ Slogan test passed')

## Module 5: Aesthetics Engine

In [ ]:
aes = score_brand(PERSONALITY, INDUSTRY, TONE, slogans[0]['text'], palette)
print(f'Overall score: {aes["overall"]}/100 — {aes["grade"]}')
for dim, score in aes['dimensions'].items():
    print(f'  {dim:25s}: {score}/100')
print('\nRecommendations:')
for r in aes['recommendations']:
    print(f'  • {r}')
assert 0 <= aes['overall'] <= 100
print('✅ Aesthetics test passed')

## Module 6: Multilingual (Fallback)

In [ ]:
translations = translate_slogan(slogans[0]['text'], COMPANY, ['Hindi','Spanish','French'])
translations = validate_translations(translations)
print(f'Translations: {len(translations)}')
for lang, data in translations.items():
    print(f'  {data["flag"]} {lang}: {data["text"][:60]} [{data["source"]}]')
assert len(translations) == 3
print('✅ Multilingual test passed')

## Module 7: Animation

In [ ]:
gif_bytes = create_brand_gif(
    logos[0]['svg'], slogans[0]['text'], palette, COMPANY, 'typewriter', 15)
print(f'GIF generated: {len(gif_bytes)/1024:.1f} KB')
assert gif_bytes[:3] == b'GIF', 'Expected GIF bytes'
print('✅ Animation test passed')

## Module 8: Export Engine

In [ ]:
zip_bytes = build_brand_kit_zip(
    company=COMPANY, industry=INDUSTRY, personality=PERSONALITY,
    logos=logos, palette=palette, fonts=fonts, slogans=slogans,
    brand_story='Test brand story.',
    translations=translations, campaigns={}, kpis={}, aesthetics=aes,
    gif_bytes=gif_bytes,
)
print(f'ZIP generated: {len(zip_bytes)/1024:.1f} KB')
assert zip_bytes[:2] == b'PK', 'Expected ZIP bytes'

import zipfile, io
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    names = z.namelist()
    print(f'Files in ZIP: {len(names)}')
    for n in names:
        print(f'  {n}')
print('✅ Export test passed')

## ✅ All Integration Tests Passed

All 8 modules tested end-to-end:
- Palette (KMeans) ✅
- Fonts (rule-based) ✅
- Logos (SVG) ✅
- Slogans (NLTK + TF-IDF) ✅
- Aesthetics (scoring) ✅
- Multilingual (fallback) ✅
- Animation (Pillow GIF) ✅
- Export (ZIP) ✅